### Programming for Science and Finance

*Prof. Götz Pfeiffer, School of Mathematical and Statistical Sciences, University of Galway*

# Notebook 5: Scientific Programming I

This notebook accompanies **Part II: Scientific Programming**. You will:

* learn to create and manipulate **NumPy arrays**, the core data structure for scientific computing in Python
* explore powerful **array generators** such as `zeros`, `arange` and `linspace`
* practice **multidimensional indexing and slicing** to access, reshape, and modify data efficiently
* apply array operations to **digital images**, understanding how pixel data can be processed numerically
* work with the **RGB colour model** and perform simple image transformations using array arithmetic
* experience the speed and elegance of **vectorized operations**, where entire datasets are processed without explicit loops
* **plot** data with `matplotlib`, as curves, points, histograms and images

By the end of this notebook, you will see how the same array-based approach supports both **scientific analysis** and **visual applications**, setting the stage for numerical simulation and data modelling in later notebooks.

## Task 1: Numpy Arrays.

When working with large amounts of data, one soon notices

* Python lists need a lot of space;
* Python loops need a lot of time.

One reason for this is Python's dynamic typing, which allows lists to contain inhomogeneous data and variables to have no fixed type.

If you know that you are working with homogeneous data, there are lots of savings to be made, both in terms of space and in terms of time.

That's what the `numpy` package promises to provide.

In [ ]:
import numpy as np
print(np.__version__)

`numpy` provides a new **data type** `np.array` for **homogeneous** lists of data.

In [ ]:
l = [3, 1, 4, 1, 5, 9, 2, 6]
a = np.array(l)
print(a)
a

Even with such a small example, the fact that an array only needs to store the common **type** of its entries once can lead to big savings in terms of space.

In [ ]:
from sys import getsizeof
print("Size of list:", getsizeof(l) + sum(getsizeof(x) for x in l))
print("Size of array:", getsizeof(a))

### From `Vector` to `np.array`.

The `Vector` class of Notebooks 3 and 4 was, in effect, a home-made version of a numpy array: a list of numbers that can be added and scaled.  Comparing the two shows what `numpy` really adds.

The `%timeit` command is a **magic command** of the notebook (magic commands start with a `%`): it runs a single line of code many times over and reports how long it takes on average.

In [ ]:
from tensor import Vector   # our own class, from Notebooks 3 and 4

n = 100000
big_v = Vector(list(range(n)))
big_w = Vector(list(range(n)))
big_a = np.arange(n)
big_b = np.arange(n)

In [ ]:
%timeit big_v + big_w   # list comprehension over python objects

In [ ]:
%timeit big_a + big_b   # one vectorized operation

The two expressions compute exactly the same sums, but the array version is faster by a factor of about $100$.  It also needs only about a quarter of the space:

In [ ]:
print("Vector:", getsizeof(big_v.data) + sum(getsizeof(x) for x in big_v.data), "bytes")
print("array :", big_a.nbytes, "bytes")

The reason is the same in both cases.  Our `Vector` stores a python list of python objects, each carrying its own type information, and `__add__` loops over them one at a time, in python.  A numpy array stores the **numbers themselves**, all of the same type, in one contiguous block of memory, and adds them in a single loop that runs in compiled C code.

So `np.array` is not a different idea from the `Vector` class -- it is the same idea, taken seriously.

`numpy` arrays can be multidimensional, e.g., 2-D matrices.

In [ ]:
ll = [list(range(i, i+4)) for i in [2,5,7]]
print(ll)

In [ ]:
aa = np.array(ll)
print(aa)

---
**Exercises.**

1. Create a $3\times 3$ matrix `A` of integers and a scalar `s = 2.5`.
Predict and verify what happens for `A + s` and `s * A`.

2. Compute the transpose of `A` as `A.T` and verify that `A.T.T` equals `A`.

3. Convert `A` to a matrix of floating point numbers via `A.astype(float)`.

4. Create and print a $3 \times 3 \times 3$ array of integers.
---

## Task 2: Array Attributes and Generators.

Each array `a` has a number of attributes, most importantly, `a.shape` and `a.dtype`.

* `a.shape`: its **length** in each dimension,
* `a.dtype`: the common **type** of all its entries.

In [ ]:
print(a.shape)
print(a.dtype)
print(aa.shape)
aa.dtype

The type of an array is fixed when the array is created.  The method `a.astype(t)` does not change it, but makes a **new** array with the same values, converted to the type `t`.

In [ ]:
print(a.dtype, a)
f = a.astype(float)
print(f.dtype, f)

Further attributes are derived from an array's shape:

* `a.ndim`: the number of dimensions (i.e., `len(a.shape)`),
* `a.size`: its total number of entries (i.e., `math.prod(a.shape)`).

In [ ]:
print(a.ndim)
print(a.size)
print(aa.ndim)
print(aa.size)

The product of a list of numbers can be computed with `math.prod`.

In [ ]:
from math import prod
prod(aa.shape)

There are a number of ways to quickly make arrays of a certain kind.
For example, an array full of **zeros**, of a given **shape** and **type**:

In [ ]:
np.zeros((3, 4), int)

Or **ones**:

In [ ]:
np.ones(10, float)

Or matrices with random entries, which can be useful, e.g. for testing purposes.
For this, you first need to make a **random number generator**, let's call it `rng`.

In [ ]:
rng = np.random.default_rng(2101)   # a fixed seed, so these examples are reproducible

With this, we can create, say, a $3 \times 4$ array of uniformly distributed random values between $0$ and $1$:

In [ ]:
rng.random((3,4))

Or a $2 \times 3 \times 4$ array of random integers in the range `range(0, 10)`:

In [ ]:
aaa = rng.integers(0, 10, (2, 3, 4))
print(aaa)

Other generators produce **evenly spaced** values.  `np.arange` works like the built-in `range`, except that it returns an array, and that it also accepts a fractional step size.  `np.linspace(start, stop, num)` produces exactly `num` values, evenly spaced from `start` to `stop`.  And `np.eye(n)` makes the $n \times n$ identity matrix.

In [ ]:
print(np.arange(1, 10))        # like range(1, 10), as an array
print(np.arange(0, 2, 0.5))    # a fractional step size
print(np.linspace(0, 1, 5))    # 5 evenly spaced values
print(np.eye(3, dtype=int))    # the 3 x 3 identity matrix

---
**Exercises.**

1. Create a $3 \times 3$ array of integers all equal to $1$.
2. Use `np.linspace` to create an array of $5$ values, evenly spaced between $1$ and $3$ (inclusive).
3. Compare `np.arange(0, 1.1, 0.2)` with `np.linspace(0, 1, 6)`.  What is the difference in how the two treat the endpoints?
4. Use `np.eye` to create a $3 \times 3$ identity matrix.
5. For `A = np.arange(12).reshape(3, 4)`, report `A.shape`, `A.ndim`, `A.size` and `A.dtype`.  Which of the four change under `A.reshape(2, 6)`, and which under `A.astype(float)`?
6. Use `np.array` to convert the `Matrix` object `ma` below into an array.  What is its `shape`?
   ```python
   from tensor import *
   ma = Matrix([Vector([1, 0, 1]), Vector([2, 1, 1]),Vector([0, 1, 1]), Vector([1, 1, 2])])
   ```

---


## Task 3: Multidimensional Indexing and Slicing.

**Recall** that `l[i]` accesses the $i$th element of a list `l`, counting from $0$.  
This kind of **indexing** can also be used to assign a new value to the $i$th element of the list `l`, as in `l[i] = 2`.  
**Negative indices** count from the end of the list, so that `l[-1]` is the last element of `l`.

In similar way, **slicing** is used to **extract a sublist** of values from a list `l`.  
In general, `l[start:stop:step]` yields the list of values from positions `start` up to (but excluding) `stop`, with step size `step`.  
Here the part `:step` is optional, with default step size $1$.  
Also, `start` can be omitted and defaults to $0$, `stop` can be omitted and defaults to `len(l)`, so that `l[:]` results in an exact copy of `l`.

In a **multi-dimensional** numpy array, elements and sublists can be accessed with the same syntax, using a single index or a slice, one for **each dimension**.

In [ ]:
print(aaa[1,1,3])
aaa[0,2,:]

### Views and Copies.

Unlike with plain python lists, array slices are **views** rather than **copies** of the array data.  
Hence, updating elements of a view will update the original array as well.

This is the array version of the difference between a **copy** and an **alias** of a list (Notebook 2), except that for an array even a *slice* is an alias.  A genuine copy is made by `a.copy()`.

In [ ]:
a2 = np.arange(6).reshape(2, 3)
row = a2[0]      # a slice: a view of the first row
row[0] = 99
print(a2)        # the original array has changed, too

In [ ]:
a3 = a2.copy()   # a genuine copy
a3[0, 0] = 0
print(a2[0, 0])  # still 99: the copy is independent

**Reshaping** is another useful operation on arrays.  As long as the `size` of an array remains the same, its elements can be refitted into any given shape.

In [ ]:
b = np.arange(1,13).reshape(3,4)
print(b)

In [ ]:
print(b.reshape(4,3))

In [ ]:
print(b.reshape(2,2,3))

### Aggregation and Sorting.

Methods like `sum()`, `mean()`, `min()` and `max()` **aggregate** the entries of an array into a single value.  In more than one dimension, the optional argument `axis` says along which dimension to aggregate: `axis=0` combines the rows, and thus yields one value per **column**, while `axis=1` combines the columns, and yields one value per **row**.

Sorting works similarly: `np.sort(a)` returns a **new** array with the entries of `a` in increasing order, whereas `np.argsort(a)` returns the **positions** of those entries in the original array -- which is what one needs in order to **rank** data.

In [ ]:
print(b)
print(b.sum(), b.mean())   # over all entries

In [ ]:
print(b.sum(axis=0))       # one sum per column
print(b.sum(axis=1))       # one sum per row

In [ ]:
scores = np.array([70, 95, 82, 61])
print(np.sort(scores))     # the values, in increasing order
print(np.argsort(scores))  # positions: the smallest sits at index 3

---
**Exercises.**

1. With `x = np.arange(1, 11)`, use slicing to select the first $5$ elements, the last $5$ elements, the elements at positions $4$, $5$ and $6$, and all the elements in reverse order.
2. Again with `x`, use slicing to select every other element starting at the beginning, every other element starting at position $1$, and every other element in reverse order starting at position $7$.
3. With the $(2, 3, 4)$-array `aaa` of Task 2, select the single entry at position $(1, 2, 3)$, the whole of row $2$ in layer $0$, and column $1$ of layer $1$.  What is the `shape` of each result?
4. Let `a2 = np.arange(6).reshape(2, 3)` and `row = a2[0]`.  Set `row[1] = 99` and print `a2`.  Repeat with `row = a2[0].copy()`.  Explain the difference.
5. With `b = np.arange(1, 13).reshape(3, 4)`, compute the sum of all entries, the sum of each column and the sum of each row.  Which `axis` gives which?  And why does `b.reshape(5, 2)` fail?
6. Use `np.argsort` on `scores` to list the positions of the scores in order of **decreasing** value -- that is, to rank them.  (Hint: a slice can reverse an array.)

---


## Task 4: Digital Image Manipulation.

* Motto: What looks like an array is an array ...
* We will treat a digital image as a numpy array of suitable shape and type.
* This will allow us to formulate (and program) certain image manipulation tasks as matrix operations.
* `Pillow` is the **Python Image Library**. Import it as `PIL`. Use mainly its `Image` class.

In [ ]:
import PIL
print(PIL.__version__)
from PIL import Image

* Use `Image.open` (on a filename) to load image data into a python session.
* Here, the image is stored as a file `"long_walk.png"` in the folder `"images"`.

In [ ]:
img = Image.open("images/long_walk.png")
img

* The image as such is not an array.

In [ ]:
type(img)

* But we can easily **convert** an image into array format ...

In [ ]:
pic = np.array(img)
print(pic.shape)
print(pic.dtype)

* The `shape` reads height, width, colour: $201$ rows of $1000$ pixels, each pixel a triple of red, green and blue intensities.  Their `dtype` `uint8` is an **unsigned $8$-bit integer**, so each intensity is a whole number between $0$ and $255$.

* ... and convert the array back to an image.

In [ ]:
Image.fromarray(pic)

---
**Exercises.**

1. What is the image corresponding to the slice `pic[:100,:100,:]`?
2. What image corresponds to `pic[:100,:100]`?
3. What image results from slicing `pic[::-1]`?
4. What is `pic[:,::-1]`?

---

## Task 5: The RGB color model.

* Interestingly, the image becomes a 3-dimensional array, that is a matrix with $201$ rows and $1000$ columns, where each entry is itself an array of length $3$.
* This corresponds to the **RGB color model**, where an image is represented as a rectangular grid (i.e., a matrix) of **pixels**.
* And each pixel is a collection of $3$ intensity values $(r, g, b)$, one for each of the colors Red, Green and Blue.
* The possible values of $r, g, b$ are the integers in the range between $0$ and $255$, that is unsigned $8$-bit integers, denoted as type `uint8` in numpy.

In [ ]:
pic[99,99]

* We can use **slicing** to crop an image, e.g., to pick just the top left corner.

In [ ]:
small = pic[:101,:201,:]
Image.fromarray(small)

* We can **separate the $3$ colors** by setting $2$ color intensities to $0$ in a **copy** of the image.  Again, slicing avoids explicit `for` loops.

In [ ]:
r = pic.copy()
r[:,:,1:] = 0
print(r[99,99])
Image.fromarray(r)

In [ ]:
b = pic.copy()
b[:,:,:2] = 0
print(b[99,99])
Image.fromarray(b)

In [ ]:
g = pic.copy()
g[:,:,::2] = 0
print(g[99,99])
Image.fromarray(g)

* Addition: note how the original image is the **sum** of `r`, `g` and `b`.

In [ ]:
Image.fromarray(r + g + b)

---
**Exercises.**

1. Separate the colors and display as grayscale images by setting

   ```python
   R, G, B = pic[:,:,0], pic[:,:,1], pic[:,:,2]
   ```
   and then convert each of these $2$-dimensional arrays into an `Image`.
2. Compute and display a grayscale version of the image as

   ```python
   0.299*R + 0.587*G + 0.114*B
   ```

3. Rebuild a colour image from the three channels with `np.stack([R, G, B], axis=2)`, and use `np.array_equal` to check that it is the original.  Then swap two of the channels, and describe what happens to the picture.

---

## Task 6: Vectorized Operations.

Note that the sum `r + g + b` of three arrays of the same shape is a `numpy` shorthand for this more explicit Python code:

In [ ]:
rgb = np.zeros(r.shape, dtype=r.dtype)
for i in range(r.shape[0]):
    for j in range(r.shape[1]):
        for k in range(r.shape[2]):
            rgb[i, j, k] = r[i, j, k] + g[i, j, k] + b[i, j, k]

Image.fromarray(rgb)

* **Scaling** is another example:  Suppose we want a darker version of `pic`, with each color intensity just half of its original value. 
In principle, all we need to do is to multiply each entry of `pic` by $0.5$.
However, the entries in `pic` have type `uint8`, and multiplying such a number with a `float` results in a `float`.

In [ ]:
p = pic[99,99,0]
print(p)
print(0.5 * p)
type(0.5 * p)

* But we can always convert this number back to the type of `pic[99, 99, 0]`, using the type `np.uint8` as a type conversion function.

In [ ]:
np.uint8(0.5 * p)

* So, here we go, looping over the pixels ...

In [ ]:
rgb = np.zeros(pic.shape, dtype=pic.dtype)
for i in range(pic.shape[0]):
    for j in range(pic.shape[1]):
        for k in range(pic.shape[2]):
            rgb[i, j, k] = np.uint8(0.5 * pic[i, j, k])

Image.fromarray(rgb)

* Again, `numpy` allows us to replace all three `for` loops by **vectorized operations** that apply to the entire array in one go, including the type conversion to `uint8`.

In [ ]:
Image.fromarray(np.uint8(0.5 * pic))

### Broadcasting.

`numpy` likes to work with element-wise operations, like matrix addition, where
the matrix sum is the matrix of the sums of corresponding entries.

How can `0.5 * pic` multiply a single number by an array of $201 \times 1000 \times 3$ entries?  `numpy` **broadcasts** the number: conceptually, it is stretched into an array of matching shape, and the multiplication is then carried out on corresponding entries.

The same happens between two arrays of **different** shapes, provided the shapes fit in the following sense.  Compare them dimension by dimension, from the right: each pair of lengths must either be equal, or one of them must be $1$, in which case that dimension is repeated as often as needed.  A missing dimension counts as $1$.  So an array of shape $(4,)$ fits every row of a $(3,4)$ array, and an array of shape $(3,1)$ fits every column.

In [ ]:
grid = np.arange(1, 13).reshape(3, 4)
print(grid)

In [ ]:
row = np.array([10, 20, 30, 40])       # shape (4,), fits a row of grid
print(grid + row)

In [ ]:
col = np.array([[100], [200], [300]])  # shape (3, 1), fits a column of grid
print(grid + col)

This is also what makes it possible to treat the three colours of an image differently in a single expression: an array of $3$ weights fits the last dimension of `pic`, and so applies one weight per colour.

In [ ]:
weights = np.array([0.5, 1.0, 1.0])    # one factor per colour
print(pic[99, 99], "->", weights * pic[99, 99])

Where an array needs an extra dimension of length $1$ to make the shapes fit, `np.newaxis` provides it.

In [ ]:
x = np.array([1, 2, 3])
print(x.shape, x[:, np.newaxis].shape)  # (3,) becomes a (3, 1) column

* Next, let's try and make a lighter version of the picture, by scaling pixels with a value bigger than $1$.

In [ ]:
Image.fromarray(np.uint8(1.5 * pic))

* The result is partially disturbed by some random noise.  What happened?
* A possible explanation is that scaling by $1.5$ can result in values larger than $255$, which **wrap around** when converted back to `uint8`: only the remainder modulo $256$ survives, so that $300$ becomes $300 - 256 = 44$, a *dark* pixel where a bright one was intended.
* Note that the wrap-around happens when `numpy` converts one of its **own** values, as in the next cell.  Handed a plain Python number, `np.uint8` refuses instead: `np.uint8(300)` raises an `OverflowError`.

In [ ]:
np.uint8(1.5 * np.uint8(200))

* In order to avoid this wrap-around, we should take care not to let the scaled value exceed $255$.

In [ ]:
np.uint8(min(1.5 * np.uint8(200), 255))

* Equipped with this technique, we can now loop over the pixels again.

In [ ]:
rgb = np.zeros(pic.shape, dtype=pic.dtype)
for i in range(pic.shape[0]):
    for j in range(pic.shape[1]):
        for k in range(pic.shape[2]):
            rgb[i, j, k] = np.uint8(min(1.5 * pic[i, j, k], 255))

Image.fromarray(rgb)

* Can this be vectorized?  Unfortunately
  ```python
      min(1.5 * pic, 255)
  ```
  does not work, and neither does
  ```python
      np.min(1.5 * pic, 255)
  ```
  (why?)  
* But `np.minimum` does work! (Check its documentation to find out how it is different from `np.min`.)

In [ ]:
Image.fromarray(np.uint8(np.minimum(1.5 * pic, 255)))

* Finally: Flip!

In [ ]:
Image.fromarray(np.flip(pic, 0))

In [ ]:
Image.fromarray(np.flip(pic, 1))

In [ ]:
Image.fromarray(np.flip(pic, 2))

In [ ]:
Image.fromarray(pic)

---
**Exercises.**

1. Use `np.flip` (or a corresponding slice) to rotate the image by 180 degrees.

2. Let `x = np.linspace(0, 2*np.pi, 1000)`. 
   Compute `np.sin(x)**2 + np.cos(x)**2` and verify it equals 1.

3. Create two arrays, an array `A` of shape $(3,1)$ and an array `B` of shape $(1,3)$, and compute
   `A + B`.  Explain the result.
---

## Task 7: Plotting with `matplotlib`.

Numbers are much easier to judge when they are drawn.  The standard plotting library for Python is **matplotlib**; by convention, its `pyplot` module is imported under the name `plt`.

The command `plt.plot(x, y)` draws a curve through the points given by two arrays `x` and `y` of the same length.  In a notebook, the figure is displayed automatically at the end of the cell.

Most plotting commands also **return** a value -- the object they have just drawn -- which the notebook would echo as a line of text underneath the figure.  To suppress that, the last command in a plotting cell is written as `_ = plt.plot(...)`: the assignment discards the value.  By convention, `_` is the name given to a value that is not needed.

In [ ]:
import matplotlib.pyplot as plt

x = np.linspace(0, 2 * np.pi, 100)
_ = plt.plot(x, np.sin(x))

Further commands **add** to the same figure, until the cell ends: several curves can be drawn on top of each other, and `xlabel`, `ylabel`, `title` and `legend` describe what is shown.  The `label` given to a curve is the text that `legend` will display for it.

In [ ]:
plt.plot(x, np.sin(x), label="sin")
plt.plot(x, np.cos(x), label="cos")
plt.xlabel("x")
plt.ylabel("value")
plt.title("Two waves")
_ = plt.legend()

Not every plot is a curve.  `plt.scatter` draws the points individually, without connecting them, and `plt.hist` sorts values into **bins** and draws how many fall into each.

In [ ]:
rng = np.random.default_rng(2101)   # a fixed seed, for reproducible plots
sample = rng.normal(size=500)

_ = plt.scatter(np.arange(len(sample)), sample, s=4)

In [ ]:
_ = plt.hist(sample, bins=20)

For arrays of pixels there is `plt.imshow`, an alternative to `Image.fromarray` that comes with axes, and that can display a $2$-dimensional array through a **colour map**, given as `cmap`.

In [ ]:
_ = plt.imshow(pic[:101, :201])

Several plots can be arranged side by side with `plt.subplots(rows, cols)`, which returns the **figure** and an array of **axes** objects, one per position.  Each axes object has its own `plot`, `imshow`, `set_title`, ... methods, and `figsize` sets the overall size of the figure, in inches.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].imshow(pic[:101, :201])
ax[0].set_title("all colours")
ax[1].imshow(pic[:101, :201, 0], cmap="gray")
_ = ax[1].set_title("the red channel")

Two further commands are worth knowing.  `plt.semilogx` (and `plt.semilogy`, and `plt.loglog`) draw the same kind of curve, but with a **logarithmic scale** on the $x$-axis (on the $y$-axis, or on both) -- useful when the values on that axis span several orders of magnitude.  And `plt.axhline(y)` and `plt.axvline(x)` draw a horizontal or a vertical **reference line** across the figure.

In [ ]:
sizes = np.array([100, 500, 1000, 5000, 10000, 50000])

plt.semilogx(sizes, 1/np.sqrt(sizes), "o-", label="1/sqrt(N)")
plt.axhline(0.01, color="gray", linestyle="--")
plt.xlabel("N")
_ = plt.legend()

On a logarithmic $N$-axis the curve is still clearly bent: $1/\sqrt{N}$ drops by a factor of $10$ only when $N$ grows by a factor of $100$.  The dashed line at $0.01$ is not reached before $N = 10^4$.  This is the rate at which Monte Carlo estimates converge (Notebook X).

Putting several of these together makes a real diagram.  Astronomers plot the stars of a
sample by **colour** against **brightness**; the result is a **Hertzsprung-Russell
diagram**, and it is the single most informative plot in stellar astronomy.

The file `data/hyg_stars.csv` holds $8000$ stars from the [HYG
catalogue](https://github.com/astronexus/HYG-Database), all of them within $100$ parsecs of
the Sun, so that their distances -- and hence their true brightnesses -- are well measured.
Two of its columns interest us here: `bv`, the **colour index** $B-V$, which is small for
blue (hot) stars and large for red (cool) ones; and `absmag`, the **absolute magnitude**,
the brightness a star would have at a standard distance.  Magnitudes run **backwards**:
smaller means brighter, and each step of $5$ is a factor of $100$ in luminosity.  So both
axes need care -- the magnitude axis is inverted, to put the luminous stars at the top.

`np.loadtxt` reads the numeric columns of a text file into an array; `usecols` picks out
the two we want, and `skiprows` steps over the header line.

In [ ]:
bv, absmag = np.loadtxt("data/hyg_stars.csv", delimiter=",",
                        skiprows=1, usecols=(1, 2), unpack=True)
print(bv.shape, absmag.shape)

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(bv, absmag, s=1, alpha=0.3)
plt.gca().invert_yaxis()          # bright stars at the top
plt.xlabel("colour index B - V")
_ = plt.ylabel("absolute magnitude")

Almost every star falls on a single diagonal band, the **main sequence**, running from hot,
blue and luminous at the top left to cool, red and faint at the bottom right.  This is where
a star spends most of its life, burning hydrogen; its position along the band is set by its
mass alone.

Two other groups stand apart.  Up and to the right, around $B-V \approx 1.0$, sits a clump
of **red giants**: cool surfaces, yet far brighter than main sequence stars of the same
colour, because they are enormously larger.  And along the bottom left is a sparse trail of
**white dwarfs**: hot, but a thousand times too faint for their colour, being the
Earth-sized remnants of stars that have finished burning.

None of this structure was put in by hand.  It is what $8000$ real stars do, and explaining
it is most of the theory of stellar evolution.

---
**Exercises.**

1. With `x = np.linspace(-2, 2, 100)`, plot $x^2$ and $x^3$ into the same figure, with a label for each curve and a legend.
2. Draw a histogram of `rng.normal(size=10000)`, with $50$ bins.  Which shape do you see?
3. Use `plt.imshow` to display the green channel of `pic` as a grayscale image.
4. Consult the documentation of `plt.savefig` and save one of your figures as a `png` file.
5. The column `dist_pc` of `data/hyg_stars.csv` holds each star's distance in parsecs; it is column $3$ for `usecols`.  Read it as well, and re-draw the Hertzsprung-Russell diagram using only the stars within $25$ parsecs.  Which of the three groups all but disappears, and why should a nearby sample be short of it?
6. Colour the points of the full diagram by distance, passing `c=dist` and `cmap="viridis"` to `plt.scatter`, and add a `plt.colorbar()`.

---


## Summary.

- NumPy arrays are the foundation of scientific programming in Python.
- Array operations replace explicit loops, leading to concise and efficient code.
- Slicing and broadcasting enable powerful manipulation of data in any dimension.
- Images are naturally represented as 3-D arrays (height × width × colour).
- Vectorization is the key to high-performance numerical computing.
- `matplotlib` turns arrays into curves, scatter plots, histograms and images.